# Testy statystyczne dla modeli

Cel notebooka:
- Wczytanie i przygotowanie zbioru
- Przeprowadzenie staryfikowanej walidacji na 10 splitach
- Porównanie wyników dwóch modeli dla metryki f1_macro
- Test Shapiro-Wilka do sprawdzenia normalności rozkładu
- Test MANN-WHITNEY do sprawdzenia identyczności rozkładu
- W zależności od wyniku testu Shapiro-Wilka wybór PAIRED T-TEST lub WILCOXON

In [1]:
from src.data_loader import load_data
from src.preprocessing import prepare_data
from src.config import MODELS
from sklearn.model_selection import StratifiedKFold
from src.statistical_tests import compare_models
import matplotlib.pyplot as plt
import numpy as np

In [2]:
df = load_data("../Dataset/WineQT.csv")

mode = "binary"

X, y = prepare_data(df, mode)

Rozkład klas (po połączeniu klas):
quality
6    621
5    522
Name: count, dtype: int64


In [3]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

rf = MODELS["RandomForestClassifier"]
cat = MODELS["CatBoost"]
dt = MODELS["DecisionTreeClassifier"]

### Test Shapiro-Wilka
- H0: Dane pochodzą z rozkładu normalnego (Należy wybrać paired T-test)
- H1: Dane nie pochodzą z rozkładu normalnego (Należy wybrać test Wilcoxona)

### Paired T-Test
- H0: Średnia różnica między modelami jest równa zero, czyli statystycznie modele są tak samo dobre
- H1: Średnia różnica między modelami nie jest równa zero, czyli statystycznie jeden model jest lepszy

### MANN-WHITNEY U
- H0: Rozkłady wyników dwóch porównywanych modeli są identyczne.
- H1: Rozkłady wyników dwóch porównywanych modeli nie są identyczne.

### Test Wilcoxona
- H0: Między wynikami dwóch modeli nie ma istotnej różnicy. Różnice w wynikach są przypadkowe.
- H1: Istnieje istotna statystycznie różnica w wynikach między dwoma modelami (jeden model radzi sobie systematycznie lepiej lub od drugiego)

In [4]:
results = compare_models(
    rf,
    cat,
    X,
    y,
    cv,
    name1="RandomForest",
    name2="CatBoost"
)


RandomForest scores:
[0.78254292 0.83296384 0.83458248 0.77016129 0.75163399 0.7646872
 0.84947573 0.71615313 0.78784119 0.81405825]

CatBoost scores:
[0.75606061 0.85054659 0.79902743 0.76167247 0.77781243 0.76734694
 0.83176699 0.68176179 0.78784119 0.80552109]

===== SHAPIRO-WILK =====
RandomForest: stat=0.9632, p=0.8212
CatBoost: stat=0.9445, p=0.6044
Nie ma podstaw do odrzucenia hipotezy zerowej

===== MANN-WHITNEY U =====
stat=54.5000, p=0.762282
Nie ma podstaw do odrzucenia hipotezy zerowej

===== PAIRED T-TEST =====
stat=1.2890, p=0.229557
Nie ma podstaw do odrzucenia hipotezy zerowej

===== WILCOXON =====
stat=11.0000, p=0.203125
Nie ma podstaw do odrzucenia hipotezy zerowej


In [5]:
results = compare_models(
    rf,
    dt,
    X,
    y,
    cv,
    name1="RandomForest",
    name2="DecisionTreeClassifier"
)


RandomForest scores:
[0.78254292 0.83296384 0.83458248 0.77016129 0.75163399 0.7646872
 0.84947573 0.71615313 0.78784119 0.81405825]

DecisionTreeClassifier scores:
[0.71304348 0.80798421 0.68502739 0.64738633 0.74401858 0.71851852
 0.76299376 0.62745098 0.7087108  0.76167247]

===== SHAPIRO-WILK =====
RandomForest: stat=0.9632, p=0.8212
DecisionTreeClassifier: stat=0.9759, p=0.9394
Nie ma podstaw do odrzucenia hipotezy zerowej

===== MANN-WHITNEY U =====
stat=88.0000, p=0.004586
Należy odrzucić hipotezę zerową

===== PAIRED T-TEST =====
stat=5.3647, p=0.000454
Należy odrzucić hipotezę zerową

===== WILCOXON =====
stat=0.0000, p=0.001953
Należy odrzucić hipotezę zerową


# Wyniki i wnioski:
- W celu przeprowadzenia testów statystycznych wykorzystano walidację krzyżową z podziałem na 10 foldów. Dla każdego modelu uzyskano 10 wyników metryki F1-score, które następnie poddano analizie statystycznej.
- Przed wyborem odpowiedniego testu statystycznego sprawdzono normalność rozkładów wyników modeli. Ponieważ w obu porównywanych przypadkach nie stwierdzono istotnych odchyleń od rozkładu normalnego, możliwe było zastosowanie parametrycznego testu t-Studenta dla prób zależnych (paired t-test).
- Pierwszy test porównywał modele Random Forest oraz CatBoost. Otrzymana wartość p-value wyniosła 0.229557, co jest wartością większą od przyjętego poziomu istotności α = 0.05. Nie ma zatem podstaw do odrzucenia hipotezy zerowej o braku różnic pomiędzy modelami.
- Dodatkowo przeprowadzono nieparametryczny test Manna–Whitneya. Wynik testu również nie wskazał na istotne różnice pomiędzy rozkładami wyników obu modeli, co potwierdza wniosek uzyskany za pomocą testu t-Studenta.
- Drugi test porównywał modele Random Forest oraz Decision Tree. Otrzymana wartość p-value wyniosła 0.000454, co jest wartością mniejszą od poziomu istotności α = 0.05. W związku z tym odrzucono hipotezę zerową i stwierdzono występowanie statystycznie istotnych różnic pomiędzy wynikami obu modeli.
- Wynik testu Manna–Whitneya również wskazał na istotne różnice pomiędzy rozkładami wyników modeli Random Forest i Decision Tree.
- Zarówno test parametryczny (paired t-test), jak i nieparametryczny (Wilcoxona) dają zgodne rezultaty w obu przypadkach.